# W4 — Ultra-long curve pairs on risk-adjusted carry, 2021–2026

> *"the ratio of the daily breakeven — the daily move in rates that yields a
> convexity gain offsetting a negative daily carry — to realized daily
> volatility. Three most attractive curves by each metric are marked in red."*
>
> — Citi, *Alert: Taking profits on delta-hedged 15y5y/20y10y flatteners*,
> close 04-Dec-2019

The brief names **risk-adjusted carry** as this book's dominant driver, and the
note above prints exactly that screen: fifteen USD curve pairs by eight
columns. This notebook takes that screen, turns it into a rule, and measures
what the rule earns.

**The verdict is that it does not work, and the reason is not costs.** Read
section 9 before section 6: the gross Sharpe is below what a zero-edge
strategy would be expected to produce from the search that found it.

In [1]:
import datetime as dt
import json
import math
import os
import pathlib
import sys

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")

REPO = pathlib.Path.cwd()
while not (REPO / "RVUtils").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd

import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"

from RVUtils.ConvexityRV import rac_backtest as B
from RVUtils.ConvexityRV import rac_signal as R
from RVUtils.ConvexityRV.strat1_threeway import expected_max_sharpe_under_null

DATA = REPO / "notebooks" / "data" / "convexity_rv"
pd.set_option("display.width", 200, "display.max_columns", 40)
print(f"repo {REPO}")

C:\Users\chris\clee\ARBS-cvx2\RVUtils\ConvexityRV\curve_ops.py:61: LicenceNotice:


Rateslib is source-available (not open-source) software distributed under a dual-licence model.
No commercial licence is registered for this installation. Use is therefore permitted for non-commercial purposes only (at-home or university based academic use).
Any use in commercial, professional, or for-profit environments, including evaluation or trial use, requires a valid commercial licence or an approved evaluation licence.
Certain features may require a registered commercial or evaluation licence in current or future versions.
For licensing information or to register a licence, please visit: https://rateslib.com/licence



repo C:\Users\chris\clee\ARBS-cvx2


## 1. The config

Every knob, and the reason for its value. `RacConfig` carries the same
defaults; they are restated here so the notebook is self-describing.

The two that matter most are `enter_pct` and `steepener_pct`. Citi publishes
**absolute** thresholds — exit around 0.8, steepener above 1.0 — and section 3
shows why they cannot be used on this curve.

In [2]:
CONFIG = R.RacConfig(
    lookback_days=756,     # 3y, matching Citi's own "3y ZS" column
    enter_pct=0.80,        # flattener when risk-adjusted carry is unusually good
    exit_pct=0.35,         # ...and closed when it reverts this far
    steepener_pct=0.20,    # the other side
    top_n=3,               # Citi marks "three most attractive" in red
    min_carry_bp=None,     # OFF: the carry SIGN is not reliable near zero (§3)
    start=dt.date(2021, 1, 1),
    end=dt.date(2026, 8, 20),
)
MIN_HOLD, MAX_HOLD = 21, 252
print(json.dumps(CONFIG.to_dict(), indent=1, default=str))

# Loaded here rather than in section 3 because the tie-out below recomputes
# from it rather than restating numbers.
panel = pd.read_parquet(DATA / "rac_screen_panel.parquet")
panel["date"] = pd.to_datetime(panel["date"])
print(f"screen panel {panel.shape}, {panel['pair'].nunique()} pairs, "
      f"{panel['date'].min().date()} .. {panel['date'].max().date()}")

{
 "lookback_days": 756,
 "enter_pct": 0.8,
 "exit_pct": 0.35,
 "steepener_pct": 0.2,
 "top_n": 3,
 "min_carry_bp": null,
 "business_days": 252.0,
 "exclude_one_sided": false,
 "start": "2021-01-01",
 "end": "2026-08-20",
 "pairs": null
}
screen panel (33690, 21), 15 pairs, 2018-01-01 .. 2026-08-20


## 2. Does the machine give the right answer to a question we already know?

Citi's 04-Dec-2019 screen prints all eight columns for all fifteen pairs —
**120 published cells** the code was never fitted to. The package's screen is
graded against them in `tests/test_convexity_rv_rac_screen.py`; the headline
numbers are restated here.

The verdict is a **split**: rank transfers, level does not.

In [3]:
# RECOMPUTED from the panel against the answer key held in the test suite, not
# restated. The published table lives in tests/test_convexity_rv_rac_screen.py
# so there is exactly one copy of it.
from scipy.stats import spearmanr

_ns: dict = {}
exec(compile((REPO / "tests" / "test_convexity_rv_rac_screen.py").read_text(
    encoding="utf-8").split("@pytest.fixture")[0], "<key>", "exec"), _ns)
PUB = {"level_bp": _ns["PUB_LEVEL"], "carry_1y_bp": _ns["PUB_CARRY"],
       "be_daily_analytic": _ns["PUB_BE"], "be_over_rv": _ns["PUB_RATIO"]}
CITI_DATE = pd.Timestamp("2019-12-04")

import RVUtils.ConvexityRV.strat3_strikeless_vol as S3

_day = (panel[panel["date"] == CITI_DATE]
        .set_index("pair").reindex([f"{s}/{l}" for s, l in S3.PAIRS_15]))
assert _day["level_bp"].notna().all(), f"the panel does not carry {CITI_DATE.date()}"

rows = []
for col, pub in PUB.items():
    if col not in _day.columns:
        continue
    ours = _day[col].to_numpy(dtype=float)
    p = np.asarray(pub, dtype=float)
    m = np.isfinite(ours) & np.isfinite(p)
    rows.append({"column": col, "n": int(m.sum()),
                 "spearman": float(spearmanr(ours[m], p[m]).statistic),
                 "mean_offset": float(np.mean(ours[m] - p[m]))})
TIE = pd.DataFrame(rows).set_index("column")
print(f"Citi {CITI_DATE.date()}, 15 pairs, recomputed from the panel:")
print(TIE.round(4).to_string())

_ratio = TIE.at["be_over_rv", "spearman"] if "be_over_rv" in TIE.index else np.nan
assert _ratio > 0.95, f"the decision statistic's RANK no longer transfers: {_ratio:.4f}"

_our_max = float(np.nanmax(_day["be_over_rv"].to_numpy(dtype=float)))
CITI_STEEPENER_THRESHOLD = 1.0
print(f"\nour max be_over_rv on that date: {_our_max:.4f}")
assert _our_max < CITI_STEEPENER_THRESHOLD, (
    f"our max ratio is {_our_max:.4f}; it now reaches Citi's 1.0 steepener "
    "threshold, so the rank-and-percentile rule may no longer be necessary"
)
print("OK: rank transfers; level does not, and Citi's 1.0 steepener threshold")
print("would fire ZERO times on our curve.")

Citi 2019-12-04, 15 pairs, recomputed from the panel:
                    n  spearman  mean_offset
column                                      
level_bp           15    0.9857      -0.8073
carry_1y_bp        15    0.9445       1.1007
be_daily_analytic  15    0.9920      -0.7521
be_over_rv         15    0.9884      -0.1947

our max be_over_rv on that date: 0.7405
OK: rank transfers; level does not, and Citi's 1.0 steepener threshold
would fire ZERO times on our curve.


## 3. Why the traded statistic is not the published one

Two measurements forced it.

**The published ratio is saturated.** Citi truncates the daily breakeven to
zero whenever carry is non-negative, so the ratio is exactly 0 on a large
fraction of the sample — worst on the tight forward pairs the factor
attribution identified as the convexity-dominated ones. A time-series
percentile of a flat-zero series is undefined exactly where the rule wants to
fire, because `carry >= 0` *is* the entry condition.

**The continuous statistic underneath it is not saturated**, is signed, passes
smoothly through zero, and is literally risk-adjusted carry:

$$\mathrm{rac} = \frac{\text{carry}_{1y}\ (\mathrm{bp})}
                       {\sigma_{\text{realised}}\ (\mathrm{bp/day}) \times \sqrt{252}}$$

In [4]:
sat = R.saturation_table(R.add_rac(panel))
print("\nsaturation of the PUBLISHED ratio, by year:")
print(sat.round(3).to_string())

assert sat["be_truncated_to_0"].max() > 0.4, "the truncation should be material"
assert (sat["rac_exactly_0"].fillna(0) == 0).all(), (
    "the continuous statistic must never be exactly zero"
)
print("\nOK: the published ratio is truncated on up to "
      f"{sat['be_truncated_to_0'].max():.1%} of a year; rac never is.")


saturation of the PUBLISHED ratio, by year:
      cells  carry>=0  be_truncated_to_0  rac_nan  rac_exactly_0
year                                                            
2018   3825     0.137              0.137    0.471            0.0
2019   3885     0.248              0.248    0.000            0.0
2020   3930     0.122              0.122    0.000            0.0
2021   3915     0.167              0.167    0.000            0.0
2022   3900     0.449              0.449    0.000            0.0
2023   3900     0.562              0.562    0.000            0.0
2024   3930     0.225              0.225    0.000            0.0
2025   3915     0.022              0.022    0.000            0.0
2026   2490     0.021              0.021    0.000            0.0

OK: the published ratio is truncated on up to 56.2% of a year; rac never is.


### 3.1 The universe is not uniform, and the module says so

A carry-keyed rule can only ever say one thing about a pair whose carry is
essentially always negative.

In [5]:
cpf = R.carry_positive_fraction(panel)
print("fraction of days with POSITIVE carry, by pair:")
print(cpf.round(3).to_string())
print(f"\none-sided family (named in the module): {list(R.ONE_SIDED_FAMILY)}")
assert cpf[list(R.ONE_SIDED_FAMILY)].max() < 0.10, (
    "the 10Yx10Y family is supposed to be structurally one-sided on this sample"
)

fraction of days with POSITIVE carry, by pair:
pair
20Yx5Y/25Yx5Y      0.542
15Yx5Y/20Yx5Y      0.435
15Yx10Y/25Yx5Y     0.408
15Yx5Y/20Yx10Y     0.378
15Yx5Y/25Yx5Y      0.371
15Yx5Y/20Yx15Y     0.315
20Yx5Y/25Yx10Y     0.311
15Yx5Y/25Yx10Y     0.266
15Yx10Y/25Yx10Y    0.236
10Yx10Y/25Yx5Y     0.027
10Yx10Y/20Yx5Y     0.025
10Yx10Y/20Yx10Y    0.022
10Yx10Y/25Yx10Y    0.018
10Yx10Y/20Yx15Y    0.013
10Yx10Y/15Yx15Y    0.009

one-sided family (named in the module): ['10Yx10Y/15Yx15Y', '10Yx10Y/20Yx5Y', '10Yx10Y/20Yx10Y', '10Yx10Y/20Yx15Y', '10Yx10Y/25Yx5Y', '10Yx10Y/25Yx10Y']


## 4. The sign probe

Re-derived from the engine on every run rather than trusted from a comment.
A flattener pays the shorter leg and receives the longer one, so the front
leg's PV01 must be positive, the back leg's negative, and the two must sum to
zero.

In [6]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

mdp = IRSwapsMDP(source="CITIVELO_EXCEL")
probe = B.sign_probe(mdp, dt.date(2022, 9, 13))
print(probe)
assert probe["is_flattener"], "bpv<0 is not a flattener on this engine"
assert abs(probe["sum"]) < 1.0, f"package is not DV01-neutral: {probe['sum']}"
assert probe["front_pv01"] > 0 > probe["back_pv01"]
print("\nOK: bpv<0 = pay front / receive back = FLATTENER, DV01-neutral to "
      f"{probe['sum']:.1e}")

{'front_pv01': 100000.00000000001, 'back_pv01': -100000.0, 'sum': 1.4551915228366852e-11, 'weights': [1.0, -1.0], 'is_flattener': True}

OK: bpv<0 = pay front / receive back = FLATTENER, DV01-neutral to 1.5e-11


C:\Users\chris\anaconda3\envs\stir\Lib\site-packages\rateslib\data\fixings.py:3426: RuntimeWarning:

invalid value encountered in divide



## 5. What this config actually trades

In [7]:
sig = R.build_signal_panel(panel, CONFIG)
_d = sig.index.get_level_values("date")
sig = sig[(_d >= pd.Timestamp(CONFIG.start)) & (_d <= pd.Timestamp(CONFIG.end))]
state = R.entry_state(sig, CONFIG)

eps = B.episodes_from_state(state, exit_pct=CONFIG.exit_pct,
                            rac_pct=sig["rac_pct"],
                            max_hold_days=MAX_HOLD, min_hold_days=MIN_HOLD)
holds = pd.Series([e.days for e in eps])
print(f"{len(eps)} episodes   median hold {holds.median():.0f} bdays   "
      f"mean {holds.mean():.0f}   max {holds.max()}")

funnel = pd.DataFrame({
    "cells (date x pair)": [len(sig)],
    "flattener days": [int((state == 1).sum())],
    "steepener days": [int((state == -1).sum())],
    "flat days": [int((state == 0).sum())],
    "episodes": [len(eps)],
    "flattener episodes": [sum(1 for e in eps if e.side == 1)],
    "steepener episodes": [sum(1 for e in eps if e.side == -1)],
}).T.rename(columns={0: "n"})
print("\n" + funnel.to_string())

by_year = (pd.DataFrame({"date": [e.entry for e in eps],
                         "side": [e.side for e in eps]})
           .assign(year=lambda d: d["date"].dt.year)
           .groupby("year")["side"]
           .agg(flat=lambda s: (s == 1).sum(), steep=lambda s: (s == -1).sum()))
print("\nepisodes entered, by year and side:")
print(by_year.to_string())
print("\nNOTE: 2024 and 2025 enter ZERO flatteners. Risk-adjusted carry declined")
print("monotonically, so on a 3y trailing window every pair sits near the bottom")
print("of its own history. The book is steepener-only for half the sample.")

117 episodes   median hold 74 bdays   mean 102   max 252

                         n
cells (date x pair)  22050
flattener days        1277
steepener days        3782
flat days            16991
episodes               117
flattener episodes      49
steepener episodes      68

episodes entered, by year and side:
      flat  steep
year             
2021    20     23
2022    14     19
2023    13     13
2024     0      6
2025     0      3
2026     2      4

NOTE: 2024 and 2025 enter ZERO flatteners. Risk-adjusted carry declined
monotonically, so on a 3y trailing window every pair sits near the bottom
of its own history. The book is steepener-only for half the sample.


## 6. How this config performed

Priced by `QueryDrivenBacktest`, daily mark-to-market. `run()` swallows
exceptions and prints them, so a failing backtest is indistinguishable from a
flat equity curve; `assert_ran` asserts on the artifacts instead.

In [8]:
ARMS = {}
for tag in ("base", "zero_cost"):
    f = DATA / f"rac_w4_equity_{tag}.parquet"
    if f.exists():
        e = pd.read_parquet(f)["equity"]
        e.index = pd.to_datetime(e.index)
        ARMS[tag] = e
print(f"loaded arms: {list(ARMS)}")
assert ARMS, "no equity curves found; run _rac_w4_run.py first"


def perf(eq: pd.Series, label: str) -> dict:
    r = eq.diff().dropna()
    span = (eq.index[-1] - eq.index[0]).days / 365.25
    return {
        "arm": label,
        "terminal": float(eq.iloc[-1]),
        "ann_sharpe": float(r.mean() / r.std() * np.sqrt(252)) if r.std() > 0 else np.nan,
        "ann_pnl": float(eq.iloc[-1] / span),
        "max_dd": float((eq - eq.cummax()).min()),
        "span_years": span,
    }


PERF = pd.DataFrame([perf(e, k) for k, e in ARMS.items()])
print(PERF.to_string(index=False))

_base = PERF[PERF["arm"] == "base"].iloc[0]
assert _base["max_dd"] < 0, "a book with no drawdown has not been marked"
print(f"\nMax drawdown is {abs(_base['max_dd']) / abs(_base['terminal']):.1f}x the "
      "terminal P&L. That is visible before any statistic is computed.")

loaded arms: ['base', 'zero_cost']
      arm     terminal  ann_sharpe      ann_pnl        max_dd  span_years
     base 9.731902e+06    0.180032 1.728039e+06 -2.381049e+07    5.631759
zero_cost 2.143190e+07    0.397393 3.805543e+06 -2.020636e+07    5.631759

Max drawdown is 2.4x the terminal P&L. That is visible before any statistic is computed.


In [9]:
_y = ARMS["base"].resample("YE").last().diff()
_y.iloc[0] = ARMS["base"].resample("YE").last().iloc[0]
print("P&L by year (base arm):")
print(_y.map(lambda v: f"{v:>14,.0f}").to_string())

P&L by year (base arm):
2021-12-31         1,942,177
2022-12-31        -9,806,208
2023-12-31         3,402,869
2024-12-31        -2,038,495
2025-12-31        12,295,082
2026-12-31         3,936,478
Freq: YE-DEC


## 7. Costs

Charged per **leg**: 2 legs x 2 sides x `half_spread_bp` on the package DV01.
At 0.25bp that is **$100,000 per closed episode**.

The only external datapoint available says that is *conservative*: Citi's own
15y5y/20y10y round trip ran **+$187K gross to +$155K net on $50K DV01**, i.e.
about **$64K per $100k DV01 including every resize**.

In [10]:
gross = float(ARMS["zero_cost"].iloc[-1])
net = float(ARMS["base"].iloc[-1])
costs = gross - net
n_ep = len(eps)
be_half = 0.25 * gross / costs if costs else np.nan

print(f"gross            {gross:>15,.0f}")
print(f"net              {net:>15,.0f}")
print(f"costs            {costs:>15,.0f}   ({costs / gross:.1%} of gross)")
print(f"per episode      {costs / n_ep:>15,.0f}")
print(f"break-even half-spread   {be_half:.3f} bp   (charged 0.250)")
print(f"  = {4 * be_half:.2f} bp round trip on the package DV01")
print(f"  Citi measured ~0.64bp; charged here 1.00bp")

assert costs > 0, "the cost arm did not charge anything"
print("\nCosts make it worse. They are NOT what makes it dead — see section 9.")

gross                 21,431,902
net                    9,731,902
costs                 11,700,000   (54.6% of gross)
per episode              100,000
break-even half-spread   0.458 bp   (charged 0.250)
  = 1.83 bp round trip on the package DV01
  Citi measured ~0.64bp; charged here 1.00bp

Costs make it worse. They are NOT what makes it dead — see section 9.


## 8. Is it convexity, or is it duration?

The `mtm_share` statistic cannot answer this — an earlier block in this package
established that by getting it wrong. Only the factor attribution answers it.

Daily P&L on the shared level / slope / curvature / convexity basis, HAC
t-stats. PC1 is 87.76% of variance with all-positive humped loadings, i.e.
level.

In [11]:
# LOADED, not typed. An earlier revision of this cell hardcoded the numbers into
# a dict and then asserted on that dict -- `assert abs(ATTR["level"]["t"]) > 3.0`
# against a literal `-4.85` cannot fail whatever the data says. Worse, the
# artifact it claimed to quote did not exist on disk at the time, because the
# script that writes it had crashed on `shares or {}` (a Series has no truth
# value) after printing. A gatekeeper pass caught both.
_attr_f = DATA / "rac_w4_attribution.json"
assert _attr_f.exists(), (
    f"{_attr_f.name} is missing. Run "
    "notebooks/backtests/convexity_rv/_rac_w4_attribution.py; this cell must "
    "read a committed artifact rather than restate numbers from prose."
)
_attr = {r["tag"]: r for r in json.loads(_attr_f.read_text())}
_base = _attr["base"]

A = pd.DataFrame({
    "share_pct": {k: 100.0 * v for k, v in _base["shares"].items()},
    "t": _base["t"],
    "incr_r2": _base["incremental_r2"],
}).reindex(["level", "slope", "curvature", "convexity", "unexplained"]).dropna(how="all")
print(A.round(4).to_string())
print(f"\ntotal R2 {_base['r2']:.4f} (adj {_base['r2_adj']:.4f}), n={_base['n_obs']:,}")
print(f"of which level supplies {_base['incremental_r2']['level']:.4f}")

_t = _base["t"]
_ir2 = _base["incremental_r2"]
assert abs(_t["level"]) > 3.0, (
    f"level t is {_t['level']:+.2f}; the claim that the significant exposure is "
    "duration no longer holds and section 12 needs re-deriving"
)
assert abs(_t["convexity"]) < 2.0, f"convexity t is {_t['convexity']:+.2f}"
assert _ir2["level"] > 10 * _ir2["convexity"], (
    f"level incremental R2 {_ir2['level']:.4f} is no longer an order of "
    f"magnitude above convexity's {_ir2['convexity']:.4f}"
)
print("\nThe only factor distinguishable from nothing is the one a DV01-neutral")
print("package is supposed not to have, and the book LOSES to it.")
print("\nCAVEAT that cuts the other way: DESIGN.md records that DAILY convexity is")
print("unidentified (t 2.1-5.4 at trade level vs 0.9-2.1 daily), so the convexity")
print("row is the expected reading at this frequency, NOT evidence of absence.")

             share_pct       t  incr_r2
level        -132.8162 -4.8525   0.1226
slope           0.1363  0.3738   0.0004
curvature       1.9990  0.1424   0.0001
convexity    -156.8272 -0.6712   0.0012
unexplained   387.5081     NaN      NaN

total R2 0.1244 (adj 0.1219), n=1,409
of which level supplies 0.1226

The only factor distinguishable from nothing is the one a DV01-neutral
package is supposed not to have, and the book LOSES to it.

CAVEAT that cuts the other way: DESIGN.md records that DAILY convexity is
unidentified (t 2.1-5.4 at trade level vs 0.9-2.1 daily), so the convexity
row is the expected reading at this frequency, NOT evidence of absence.


## 9. What the search cost — and why this is dead

**Read this before section 6.** Positions are held a mean of ~102 business
days, so 117 episodes over 5.63 years are not 117 independent observations.

In [12]:
mean_hold = float(holds.mean())
span = float(PERF[PERF["arm"] == "base"]["span_years"].iloc[0])
n_eff = span * 252.0 / mean_hold
print(f"mean hold {mean_hold:.0f} bdays -> n_eff ~ {n_eff:.1f} independent holds")

rows = []
for N in (1, 6, 12, 24, 48):
    rows.append({"trials": N,
                 "E[max SR | null]": expected_max_sharpe_under_null(N, n_obs=int(n_eff))})
NULL = pd.DataFrame(rows)
print("\n" + NULL.round(3).to_string(index=False))

gross_sr = float(PERF[PERF["arm"] == "zero_cost"]["ann_sharpe"].iloc[0])
net_sr = float(PERF[PERF["arm"] == "base"]["ann_sharpe"].iloc[0])
null_12 = float(NULL[NULL["trials"] == 12]["E[max SR | null]"].iloc[0])
print(f"\ngross Sharpe {gross_sr:.3f}   net {net_sr:.3f}")
print(f"E[max SR | null] at 12 trials: {null_12:.3f}")

assert gross_sr < null_12, (
    "the gross Sharpe now clears the 12-trial null; the verdict in this "
    "notebook needs re-deriving"
)
print("\nVERDICT: the GROSS Sharpe is below the null expectation for a 12-cell")
print("search, and the exit-rule x minimum-hold sweep alone was 12 cells.")
print("No cost assumption rescues this, because the failure is in the gross number.")

mean hold 102 bdays -> n_eff ~ 14.0 independent holds

 trials  E[max SR | null]
      1             0.000
      6             0.361
     12             0.462
     24             0.549
     48             0.627

gross Sharpe 0.397   net 0.180
E[max SR | null] at 12 trials: 0.462

VERDICT: the GROSS Sharpe is below the null expectation for a 12-cell
search, and the exit-rule x minimum-hold sweep alone was 12 cells.
No cost assumption rescues this, because the failure is in the gross number.


## 10. Robustness

The equity path, and the drawdown that the Sharpe alone does not convey.

In [13]:
from BT.trade_dashboard import compare_curves

try:
    fig = compare_curves({k: v for k, v in ARMS.items()}, span_years=span)
    fig.show()
except Exception as exc:                                          # noqa: BLE001
    print(f"compare_curves unavailable ({exc}); plotting directly")
    import plotly.graph_objects as go
    fig = go.Figure()
    for k, v in ARMS.items():
        fig.add_trace(go.Scatter(x=v.index, y=v.to_numpy(), name=k, mode="lines"))
    fig.update_layout(title="W4 equity, base vs zero cost",
                      yaxis_title="cumulative MTM, USD", height=420)
    fig.show()

compare_curves unavailable (to_book() got an unexpected keyword argument 'span_years'); plotting directly


In [14]:
dd = ARMS["base"] - ARMS["base"].cummax()
print(f"time in drawdown: {float((dd < 0).mean()):.1%} of days")
print(f"worst drawdown  : {float(dd.min()):,.0f}")
print(f"terminal        : {float(ARMS['base'].iloc[-1]):,.0f}")

time in drawdown: 97.6% of days
worst drawdown  : -23,810,491
terminal        : 9,731,902


## 11. Trade log

In [15]:
LOG = pd.DataFrame([{
    "pair": e.pair,
    "side": "flattener" if e.side == 1 else "steepener",
    "entry": e.entry.date(), "exit": e.exit.date(), "bdays": e.days,
} for e in eps]).sort_values("entry")
print(f"{len(LOG)} episodes")
print(LOG.head(20).to_string(index=False))
print("...")
print(LOG.groupby(["pair", "side"]).size().unstack(fill_value=0).to_string())

117 episodes
           pair      side      entry       exit  bdays
15Yx10Y/25Yx10Y steepener 2021-01-01 2021-03-01     41
  15Yx5Y/20Yx5Y flattener 2021-01-01 2021-07-08    134
 15Yx5Y/20Yx10Y flattener 2021-01-01 2021-02-26     40
  15Yx5Y/25Yx5Y flattener 2021-01-01 2021-02-26     40
 20Yx5Y/25Yx10Y steepener 2021-01-01 2021-03-01     41
10Yx10Y/25Yx10Y steepener 2021-01-04 2021-02-17     32
 10Yx10Y/25Yx5Y flattener 2021-01-04 2021-06-15    116
  20Yx5Y/25Yx5Y steepener 2021-02-04 2021-06-21     97
10Yx10Y/15Yx15Y flattener 2021-02-17 2021-06-11     82
10Yx10Y/20Yx10Y flattener 2021-02-17 2021-06-18     87
 10Yx10Y/20Yx5Y flattener 2021-02-17 2021-06-18     87
10Yx10Y/20Yx15Y flattener 2021-02-26 2021-06-14     76
 15Yx10Y/25Yx5Y steepener 2021-02-26 2021-03-29     21
10Yx10Y/25Yx10Y flattener 2021-02-26 2021-05-12     53
 20Yx5Y/25Yx10Y flattener 2021-03-01 2021-03-30     21
15Yx10Y/25Yx10Y flattener 2021-03-01 2021-05-07     49
 15Yx5Y/20Yx10Y flattener 2021-03-08 2021-07-08     

## 12. Reading this notebook

* **The result is a negative one and it is not about costs.** Section 9 is the
  verdict: the gross Sharpe sits below what a zero-edge strategy would be
  expected to produce from the search that found it. Costs (55% of gross) only
  make it worse, and the cost charged here is already more conservative than
  the one external datapoint available.

* **The screen underneath it is sound.** 120 published cells reproduce in
  rank (spearman 0.988 on the decision statistic). What does not transfer is
  the *level*, and Citi's absolute thresholds with it — which is why the rule
  is keyed on rank and percentile.

* **Section 8 says it is duration.** The only factor distinguishable from
  noise is level, at t = −4.85, and the book loses to it. The convexity row is
  not evidence of absence at daily frequency; the verdict does not rest on it.

* **What would have to change.** Not the thresholds — the search is what
  killed it. It needs fewer, larger, longer holds so `n_eff` rises, or a hedge
  that removes the drawdown rather than the return. `factor_neutral_sizing`
  already computes a PC1 hedge and is the obvious next thing to try.

Full write-up: `docs/convexityrv/results/w4-ultra-long-risk-adjusted-carry.md`.